# House Price Prediction Using Machine Learning

**Author:** Thankgod Philemon Akor

This notebook develops an end-to-end regression workflow on the Ames Housing dataset and compares:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

## 1. Imports and Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
np.random.seed(42)

DATA_PATH = Path("../data/train.csv")

## 2. Load and Inspect the Data

In [ ]:
house_price = pd.read_csv(DATA_PATH)

display(house_price.head())
print("Rows, Columns:", house_price.shape)
house_price.info()
display(house_price.describe())
display(house_price.describe(include="object"))
print("Duplicate rows:", house_price.duplicated().sum())

## 3. Missing-Value Review and Domain-Aware Cleaning

The Ames dataset uses some missing values to represent the **absence of an amenity**.  
Examples include houses with no pool, no garage, no basement, or no fireplace.

Those cases are treated differently from genuinely unknown values.

In [ ]:
plt.figure(figsize=(20, 7))
sns.heatmap(house_price.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Missing Values")
plt.show()

missing_percent = (house_price.isnull().mean() * 100).sort_values(ascending=False)
display(missing_percent[missing_percent > 0])

In [ ]:
# Read again without automatic NA conversion to distinguish literal "NA"/"None"
# from values that pandas interprets as missing.
data_check = pd.read_csv(DATA_PATH, keep_default_na=False)

# Domain-based categorical replacements
house_price["PoolQC"] = house_price["PoolQC"].fillna("No Pool")
house_price["MiscFeature"] = house_price["MiscFeature"].fillna("None")
house_price["Alley"] = house_price["Alley"].fillna("No Alley")
house_price["Fence"] = house_price["Fence"].fillna("No Fence")
house_price["FireplaceQu"] = house_price["FireplaceQu"].fillna("No Fireplace")

# Masonry veneer type: preserve legitimate 'None' and label genuinely missing values.
valid_mask = data_check["MasVnrType"] == "None"
missing_mask = data_check["MasVnrType"] == "NA"
house_price.loc[valid_mask, "MasVnrType"] = "None"
house_price.loc[missing_mask, "MasVnrType"] = np.nan
house_price["MasVnrType"] = house_price["MasVnrType"].fillna("Unknown")

# Median among houses with masonry veneer
mas_vnr_area_median = house_price.loc[
    (house_price["MasVnrType"] != "None") &
    (house_price["MasVnrType"].notna()),
    "MasVnrArea"
].median()
house_price["MasVnrArea"] = house_price["MasVnrArea"].fillna(mas_vnr_area_median)

# Lot frontage: use the neighborhood median
lot_frontage_neighborhood_median = (
    house_price.groupby("Neighborhood")["LotFrontage"].median()
)
house_price["LotFrontage"] = house_price["LotFrontage"].fillna(
    house_price["Neighborhood"].map(lot_frontage_neighborhood_median)
)

# Garage-related missing values
garage_cat_cols = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
garage_num_cols = ["GarageCars", "GarageArea", "GarageYrBlt"]

for col in garage_cat_cols:
    house_price[col] = house_price[col].fillna("No Garage")

for col in garage_num_cols:
    house_price[col] = house_price[col].fillna(0)

# Basement-related missing values
basement_cat_cols = [
    "BsmtFinType2", "BsmtExposure", "BsmtCond", "BsmtQual", "BsmtFinType1"
]

for col in basement_cat_cols:
    house_price[col] = house_price[col].fillna("No Basement")

house_price["BsmtExposure"] = house_price["BsmtExposure"].replace(
    "No", "No Exposure"
)

# Electrical: fill the one missing category using the mode
house_price["Electrical"] = house_price["Electrical"].fillna(
    house_price["Electrical"].mode()[0]
)

remaining_missing = house_price.isnull().sum()
print(remaining_missing[remaining_missing > 0])

> **Portfolio note:** the domain-based replacements above are reasonable, but the statistically learned values
> (medians and mode) are computed before the train/test split in this original project.
> A future version should fit those statistics on the training set only.

## 4. Basic Type Cleanup

In [ ]:
house_price_cleaned = house_price.copy()

# Numeric codes that represent categories
house_price_cleaned["MSSubClass"] = house_price_cleaned["MSSubClass"].astype(str)
house_price_cleaned["MoSold"] = house_price_cleaned["MoSold"].astype(str)

print("Shape:", house_price_cleaned.shape)

## 5. Target Distribution and Outlier Investigation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(house_price_cleaned["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of SalePrice")

sns.boxplot(x=house_price_cleaned["SalePrice"], ax=axes[1])
axes[1].set_title("Boxplot of SalePrice")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x="GrLivArea",
    y="SalePrice",
    data=house_price_cleaned
)
plt.title("GrLivArea vs SalePrice")
plt.xlabel("Above Ground Living Area")
plt.ylabel("Sale Price")
plt.show()

suspected_outliers = house_price_cleaned[
    (house_price_cleaned["GrLivArea"] > 4000) &
    (house_price_cleaned["SalePrice"] < 300000)
]

display(
    suspected_outliers[
        ["GrLivArea", "SalePrice", "OverallQual", "OverallCond",
         "YearBuilt", "SaleCondition"]
    ]
)

In [ ]:
# Remove the two influential observations identified above.
house_price_model = house_price_cleaned.drop(
    suspected_outliers.index
).copy()

print("Before:", house_price_cleaned.shape)
print("After:", house_price_model.shape)

## 6. Exploratory Data Analysis

In [ ]:
numeric_data = house_price_model.select_dtypes(include="number")

saleprice_corr = numeric_data.corr()["SalePrice"].sort_values(ascending=False)
display(saleprice_corr.head(15))

top_correlations = saleprice_corr.abs().sort_values(ascending=False).head(15)
top_corr_cols = top_correlations.index

plt.figure(figsize=(12, 8))
sns.heatmap(
    house_price_model[top_corr_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Heatmap of Top Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
top_features = [
    "OverallQual", "GrLivArea", "TotalBsmtSF", "GarageCars", "GarageArea"
]

for col in top_features:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=col, y="SalePrice", data=house_price_model)
    plt.title(f"{col} vs SalePrice")
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(
    x="Neighborhood",
    y="SalePrice",
    data=house_price_model
)
plt.title("SalePrice by Neighborhood")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

neighborhood_price = (
    house_price_model.groupby("Neighborhood")["SalePrice"]
    .median()
    .sort_values(ascending=False)
)
display(neighborhood_price)

## 7. Feature Engineering

In [ ]:
house_price_model["TotalSF"] = (
    house_price_model["TotalBsmtSF"] +
    house_price_model["1stFlrSF"] +
    house_price_model["2ndFlrSF"]
)

house_price_model["TotalBathrooms"] = (
    house_price_model["FullBath"] +
    0.5 * house_price_model["HalfBath"] +
    house_price_model["BsmtFullBath"] +
    0.5 * house_price_model["BsmtHalfBath"]
)

house_price_model["HouseAge"] = (
    house_price_model["YrSold"] - house_price_model["YearBuilt"]
)

house_price_model["RemodAge"] = (
    house_price_model["YrSold"] - house_price_model["YearRemodAdd"]
)

house_price_model["HasGarage"] = (house_price_model["GarageArea"] > 0).astype(int)
house_price_model["HasBasement"] = (house_price_model["TotalBsmtSF"] > 0).astype(int)
house_price_model["HasFireplace"] = (house_price_model["Fireplaces"] > 0).astype(int)
house_price_model["HasPool"] = (house_price_model["PoolArea"] > 0).astype(int)

## 8. Log-Transform the Target

In [ ]:
house_price_model["LogSalePrice"] = np.log1p(house_price_model["SalePrice"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(house_price_model["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("Original SalePrice")

sns.histplot(house_price_model["LogSalePrice"], kde=True, ax=axes[1])
axes[1].set_title("Log-Transformed SalePrice")

plt.tight_layout()
plt.show()

print("Original skewness:", house_price_model["SalePrice"].skew())
print("Log-transformed skewness:", house_price_model["LogSalePrice"].skew())

## 9. Train/Test Split

In [ ]:
X = house_price_model.drop(columns=["SalePrice", "LogSalePrice", "Id"])
y = house_price_model["LogSalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 10. Preprocessing Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

ordinal_features = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual",
    "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond"
]

nominal_features = [
    col for col in categorical_features
    if col not in ordinal_features
]

print("Numerical:", len(numeric_features))
print("Ordinal:", len(ordinal_features))
print("Nominal:", len(nominal_features))
print("Total:", len(numeric_features) + len(ordinal_features) + len(nominal_features))

In [ ]:
ordinal_orders = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["No Basement", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["No Basement", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["No Basement", "No Exposure", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["No Basement", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["No Basement", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["No Fireplace", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish": ["No Garage", "Unf", "RFn", "Fin"],
    "GarageQual": ["No Garage", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["No Garage", "Po", "Fa", "TA", "Gd", "Ex"],
}

ordinal_categories = [
    ordinal_orders[col] for col in ordinal_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "ord",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            ordinal_features
        ),
        (
            "nom",
            OneHotEncoder(handle_unknown="ignore"),
            nominal_features
        )
    ]
)

## 11. Build and Train the Models

In [ ]:
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

linear_pipeline.fit(X_train, y_train)
decision_tree_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

## 12. Evaluate on the Holdout Test Set

In [ ]:
def evaluate_pipeline(name, pipeline, X_test, y_test):
    pred_log = pipeline.predict(X_test)
    pred_price = np.expm1(pred_log)
    true_price = np.expm1(y_test)

    mae = mean_absolute_error(true_price, pred_price)
    rmse = np.sqrt(mean_squared_error(true_price, pred_price))
    r2 = r2_score(true_price, pred_price)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }, pred_price, true_price

linear_result, linear_pred_price, y_test_price = evaluate_pipeline(
    "Linear Regression", linear_pipeline, X_test, y_test
)
tree_result, dt_pred_price, _ = evaluate_pipeline(
    "Decision Tree", decision_tree_pipeline, X_test, y_test
)
rf_result, rf_pred_price, _ = evaluate_pipeline(
    "Random Forest", random_forest_pipeline, X_test, y_test
)

results = pd.DataFrame([linear_result, tree_result, rf_result])
display(results.round(4))

In [ ]:
results_plot = results.set_index("Model")[["MAE", "RMSE"]]
results_plot.plot(kind="bar", figsize=(9, 5))
plt.title("Model Comparison: MAE and RMSE")
plt.ylabel("Error ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test_price, y=linear_pred_price)

min_price = min(y_test_price.min(), linear_pred_price.min())
max_price = max(y_test_price.max(), linear_pred_price.max())

plt.plot(
    [min_price, max_price],
    [min_price, max_price],
    linestyle="--"
)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

## 13. Five-Fold Cross-Validation

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

models = {
    "Linear Regression": linear_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline
}

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    cv_results.append({
        "Model": name,
        "CV_MAE": -scores["test_MAE"].mean(),
        "CV_RMSE": -scores["test_RMSE"].mean(),
        "CV_R2": scores["test_R2"].mean(),
        "R2_STD": scores["test_R2"].std()
    })

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df.round(4))

## 14. Practical Prediction Check on 20 Unseen Houses

In [ ]:
sample_houses = X_test.sample(n=20, random_state=42)
actual_prices = np.expm1(y_test.loc[sample_houses.index])

linear_pred_price = np.expm1(linear_pipeline.predict(sample_houses))
dt_pred_price = np.expm1(decision_tree_pipeline.predict(sample_houses))
rf_pred_price = np.expm1(random_forest_pipeline.predict(sample_houses))

prediction_comparison = pd.DataFrame({
    "Actual_Price": actual_prices.values,
    "Linear_Regression": linear_pred_price,
    "Decision_Tree": dt_pred_price,
    "Random_Forest": rf_pred_price
}).round(2)

prediction_comparison["Linear_Error"] = (
    prediction_comparison["Actual_Price"] -
    prediction_comparison["Linear_Regression"]
).abs()

prediction_comparison["DecisionTree_Error"] = (
    prediction_comparison["Actual_Price"] -
    prediction_comparison["Decision_Tree"]
).abs()

prediction_comparison["RandomForest_Error"] = (
    prediction_comparison["Actual_Price"] -
    prediction_comparison["Random_Forest"]
).abs()

display(prediction_comparison.round(2))

print("Average absolute error:")
print("Linear Regression:", round(prediction_comparison["Linear_Error"].mean(), 2))
print("Decision Tree:", round(prediction_comparison["DecisionTree_Error"].mean(), 2))
print("Random Forest:", round(prediction_comparison["RandomForest_Error"].mean(), 2))

## 15. Conclusion

The three-model comparison shows that:
- Linear Regression is a strong baseline and performs very competitively.
- A single Decision Tree generalizes substantially worse.
- Random Forest delivers the best holdout RMSE and R² and performs strongly on unseen-house checks.

The next step is to eliminate the remaining leakage risk in statistically learned imputations and then tune the strongest models.